# DemandPulse
## Hierarchical Forecasting & Forecast Reconciliation

**Notebook 04 — Coherent Multi-Level Retail Demand Forecasting**

This notebook continues the DemandPulse research program by asking a different forecasting question:

> **Can forecasts made independently at multiple organizational levels be reconciled so that they are mathematically coherent without sacrificing too much predictive accuracy?**

The hierarchy used here is strictly nested:

`Company → City → Store → Store-local Category → SKU`

where the category node uses `third_category_id` **within each store**, and the bottom series is the same `(store_id, product_id)` identity used in earlier DemandPulse notebooks.

The notebook compares:

- independent/unreconciled LightGBM forecasts,
- bottom-up reconciliation,
- top-down reconciliation,
- store-level middle-out reconciliation,
- a practical MinT-style weighted least-squares reconciliation.

All evaluation is chronological. The official Hugging Face `eval` split is not used to estimate reconciliation weights, historical proportions, or model parameters.

## 1. Environment

Only packages used by this notebook are installed. `scipy` is required for sparse hierarchy matrices and the iterative MinT-style linear solve.

In [ ]:
# %pip install -q "datasets>=3.0" "pandas>=2.0" "numpy>=1.24" "pyarrow>=14" \
    "matplotlib>=3.7" "scikit-learn>=1.3" "lightgbm>=4.0" "tqdm>=4.66" "scipy>=1.10"

In [ ]:
import gc
import hashlib
import json
import platform
import random
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import datasets
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from datasets import load_dataset
from IPython.display import Markdown, display
from scipy import sparse
from scipy.sparse.linalg import LinearOperator, cg
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.auto import tqdm

print("Python:", platform.python_version())
print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn: imported metrics API")
print("LightGBM:", lgb.__version__)
print("SciPy:", scipy.__version__)

## 2. Project configuration

`SERIES_SAMPLE_PERCENT` uses the **same complete-series SHA-256 rule as Notebook 01**. Setting it to `100` preserves the full hierarchy at substantially higher computational cost.

In [ ]:
DATASET_ID = "Dingdong-Inc/FreshRetailNet-50K"
RANDOM_STATE = 42
SERIES_SAMPLE_PERCENT = 30
SERIES_KEYS = ["store_id", "product_id"]
FORECAST_HORIZON = 7
N_BACKTEST_FOLDS = 3
STREAM_BATCH_SIZE = 50_000
PROPORTION_WINDOW_DAYS = 28
MIDDLE_OUT_LEVEL = "Store"
MIN_HISTORY_DAYS = 28
COHERENCE_TOLERANCE = 1e-8
MINT_RIDGE = 1e-6
RUN_FULL_BACKTEST = True

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

assert 0 < SERIES_SAMPLE_PERCENT <= 100
assert FORECAST_HORIZON >= 1
assert N_BACKTEST_FOLDS >= 1
assert MIDDLE_OUT_LEVEL == "Store"

REPORTS_DIR = Path("reports")
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = Path("models")
HIER_MODELS_DIR = MODELS_DIR / "hierarchical"
for path in (REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, HIER_MODELS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ID}")
print(f"Series sample: {SERIES_SAMPLE_PERCENT}%")
print(f"Forecast horizon: {FORECAST_HORIZON} days")
print(f"Backtest folds: {N_BACKTEST_FOLDS}")

## 3. Prior-notebook artifact continuity

Notebook 04 runs independently after a fresh kernel restart. Earlier DemandPulse artifacts are loaded **when available** to preserve research continuity, but they are treated as read-only context and are not required for hierarchical model fitting.

This avoids hidden dependence on variables from Notebooks 01–03.

In [ ]:
PREVIOUS_ARTIFACTS = {
    "Notebook 01 final comparison": TABLES_DIR / "final_model_comparison.csv",
    "Notebook 01 baseline backtest": TABLES_DIR / "baseline_backtest_results.csv",
    "Notebook 02 probabilistic metrics": TABLES_DIR / "probabilistic_final_metrics.csv",
    "Notebook 03 causal ATE": TABLES_DIR / "ate_estimates.csv",
}

previous_results: Dict[str, pd.DataFrame] = {}
for label, path in PREVIOUS_ARTIFACTS.items():
    if path.exists():
        previous_results[label] = pd.read_csv(path)
        print(f"Loaded {label}: {path}")
        display(previous_results[label].head())
    else:
        print(f"Warning: {label} not found at {path}; continuing without it.")

## 4. Remote data source and schema validation

FreshRetailNet-50K is accessed programmatically with Hugging Face streaming. Streaming returns an `IterableDataset`; the source is therefore consumed progressively rather than treated as a random-access pandas table.

Only columns needed for the hierarchy are materialized. Same-day hourly sequence fields are not needed here.

In [ ]:
REQUIRED_COLUMNS = [
    "city_id",
    "store_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "product_id",
    "dt",
    "sale_amount",
    "stock_hour6_22_cnt",
    "discount",
    "holiday_flag",
    "activity_flag",
]

MATERIALIZED_COLUMNS = [
    "city_id",
    "store_id",
    "third_category_id",
    "product_id",
    "dt",
    "sale_amount",
]

ROW_KEY = ["store_id", "product_id", "dt"]

train_stream = load_dataset(DATASET_ID, split="train", streaming=True)
eval_stream = load_dataset(DATASET_ID, split="eval", streaming=True)


def validate_stream_schema(stream, split_name: str) -> List[str]:
    """Assert that the remote iterable exposes every required field."""
    if getattr(stream, "features", None) is not None:
        returned = list(stream.features.keys())
    else:
        returned = list(next(iter(stream)).keys())
    missing = sorted(set(REQUIRED_COLUMNS) - set(returned))
    assert not missing, f"{split_name} missing required fields: {missing}. Returned: {returned}"
    return returned

train_columns = validate_stream_schema(train_stream, "train")
eval_columns = validate_stream_schema(eval_stream, "eval")
assert set(train_columns) == set(eval_columns), "Train/eval remote schemas differ."
print("Remote schema validated.")

### Deterministic complete-series sampling

The exact rule from Notebook 01 is preserved: hash the UTF-8 token `store_id-product_id` with SHA-256, convert the first eight digest bytes to an integer, take modulo 10,000, and compare with `percent × 100`.

This selects whole time series rather than arbitrary rows and applies the same selection to train and eval.

In [ ]:
def series_selected(store_id, product_id, percent: float, cache: Dict[Tuple, bool]) -> bool:
    """Notebook-01-compatible deterministic series selector."""
    if percent >= 100:
        return True
    key = (store_id, product_id)
    if key not in cache:
        token = f"{store_id}-{product_id}".encode("utf-8")
        digest = hashlib.sha256(token).digest()
        bucket = int.from_bytes(digest[:8], byteorder="big", signed=False) % 10_000
        cache[key] = bucket < int(round(percent * 100))
    return cache[key]


def materialize_stream(stream, split_name, sample_percent, batch_size=50_000):
    """
    Materialize a deterministic subset of complete store-product series.
    """
    import time
    if not hasattr(stream, "batch"):
        raise RuntimeError(
            "This notebook expects a recent Hugging Face datasets version with IterableDataset.batch(). "
            "Re-run the install cell and restart the kernel if necessary."
        )

    frames = []
    selection_cache = {}
    selected_missing_counts = {c: 0 for c in REQUIRED_COLUMNS}
    rows_processed = 0
    rows_selected = 0

    max_retries = 5
    for attempt in range(max_retries):
        try:
            frames.clear()
            rows_processed = 0
            rows_selected = 0
            selected_missing_counts = {c: 0 for c in REQUIRED_COLUMNS}
            batched_stream = stream.batch(batch_size=batch_size)
            for batch in tqdm(batched_stream, desc=f"Streaming {split_name}", unit="batch"):
                batch_df = pd.DataFrame(batch)
                rows_processed += len(batch_df)

                keep_mask = np.fromiter(
                    (
                        series_selected(store_id, product_id, sample_percent, selection_cache)
                        for store_id, product_id in zip(batch_df["store_id"], batch_df["product_id"])
                    ),
                    dtype=bool,
                    count=len(batch_df),
                )
                if not keep_mask.any():
                    continue

                selected = batch_df.loc[keep_mask]
                rows_selected += len(selected)
                for column in REQUIRED_COLUMNS:
                    selected_missing_counts[column] += int(selected[column].isna().sum())

                frames.append(selected[MATERIALIZED_COLUMNS].copy())
            break
        except Exception as err:
            if attempt == max_retries - 1:
                raise err
            print(f"Network glitch during {split_name} streaming (attempt {attempt + 1}/{max_retries}): {err}. Retrying in 3s...")
            time.sleep(3)

    if not frames:
        raise RuntimeError(
            f"No rows were selected from {split_name}. Check SERIES_SAMPLE_PERCENT and source availability."
        )

    result = pd.concat(frames, ignore_index=True, copy=False)
    result.attrs["rows_processed"] = rows_processed
    result.attrs["rows_selected"] = rows_selected
    result.attrs["selected_missing_counts"] = selected_missing_counts
    result.attrs["schema_types"] = train_schema_types if split_name == "train" else eval_schema_types

    print(
        f"{split_name}: processed {rows_processed:,} rows; "
        f"selected {rows_selected:,} rows ({rows_selected / rows_processed:.1%})."
    )
    return result

In [ ]:
train_df = materialize_stream(train_stream, "train", SERIES_SAMPLE_PERCENT, STREAM_BATCH_SIZE)
eval_df = materialize_stream(eval_stream, "eval", SERIES_SAMPLE_PERCENT, STREAM_BATCH_SIZE)

for name, frame in (("train", train_df), ("eval", eval_df)):
    frame["dt"] = pd.to_datetime(frame["dt"], errors="raise")
    print(
        f"{name} shape={frame.shape}; memory={frame.memory_usage(deep=True).sum()/1024**2:.2f} MiB; "
        f"dates={frame['dt'].min().date()} to {frame['dt'].max().date()}"
    )

## 5. Data integrity and temporal holdout audit

Duplicate bottom-level keys are displayed before execution stops. The notebook does not silently delete duplicates or repair temporal overlap.

In [ ]:
def validate_bottom_data(df: pd.DataFrame, split_name: str) -> None:
    missing = sorted(set(MATERIALIZED_COLUMNS) - set(df.columns))
    assert not missing, f"{split_name} materialization missing columns: {missing}"

    duplicate_mask = df.duplicated(ROW_KEY, keep=False)
    print(f"{split_name} duplicate bottom keys: {int(duplicate_mask.sum()):,}")
    if duplicate_mask.any():
        display(df.loc[duplicate_mask].sort_values(ROW_KEY).head(30))
        raise ValueError(f"{split_name} contains duplicate (store_id, product_id, dt) rows.")

    if df["sale_amount"].isna().any():
        raise ValueError(f"{split_name} contains missing sale_amount values.")
    if df["sale_amount"].lt(0).any():
        raise ValueError(f"{split_name} contains negative sale_amount values.")

validate_bottom_data(train_df, "train")
validate_bottom_data(eval_df, "eval")

train_min, train_max = train_df["dt"].min(), train_df["dt"].max()
eval_min, eval_max = eval_df["dt"].min(), eval_df["dt"].max()
STRICT_TEMPORAL_HOLDOUT = train_max < eval_min

print("Train:", train_min, "→", train_max)
print("Eval:", eval_min, "→", eval_max)
print("Strict temporal holdout:", STRICT_TEMPORAL_HOLDOUT)

if not STRICT_TEMPORAL_HOLDOUT:
    raise ValueError(
        f"Official eval is not strictly later than train: train_max={train_max}, eval_min={eval_min}. "
        "Hierarchical final evaluation is stopped to prevent leakage."
    )

## 6. Constructing a strictly nested retail hierarchy

To make reconciliation mathematically well-defined, every bottom SKU series must map to exactly one city and one `third_category_id` during training.

The category node is **store-local**:

`City / Store / third_category_id`

This avoids pretending that a retailer-wide category total lies between store and product in a strict tree.

In [ ]:
def validate_static_hierarchy_mapping(df: pd.DataFrame) -> pd.DataFrame:
    """Return one static hierarchy mapping row per bottom series after validating nesting."""
    mapping_counts = (
        df.groupby(SERIES_KEYS, observed=True)[["city_id", "third_category_id"]]
        .nunique(dropna=False)
    )
    invalid = mapping_counts[(mapping_counts > 1).any(axis=1)]
    if not invalid.empty:
        display(invalid.head(20))
        raise ValueError(
            "Some (store_id, product_id) series change city/category over time, so the requested strict hierarchy "
            "is not valid without a time-varying hierarchy design."
        )

    store_city_counts = df.groupby("store_id", observed=True)["city_id"].nunique(dropna=False)
    bad_stores = store_city_counts[store_city_counts > 1]
    if not bad_stores.empty:
        display(bad_stores.head(20))
        raise ValueError("Some stores map to multiple cities; strict nesting is invalid.")

    return (
        df[["city_id", "store_id", "third_category_id", "product_id"]]
        .drop_duplicates()
        .sort_values(["city_id", "store_id", "third_category_id", "product_id"])
        .reset_index(drop=True)
    )

train_mapping = validate_static_hierarchy_mapping(train_df)
eval_mapping = validate_static_hierarchy_mapping(eval_df)

train_bottom_keys = set(map(tuple, train_mapping[SERIES_KEYS].to_numpy()))
eval_bottom_keys = set(map(tuple, eval_mapping[SERIES_KEYS].to_numpy()))
existing_bottom = train_bottom_keys & eval_bottom_keys
cold_start_bottom = eval_bottom_keys - train_bottom_keys
train_only_bottom = train_bottom_keys - eval_bottom_keys

print(f"Train bottom series: {len(train_bottom_keys):,}")
print(f"Eval bottom series: {len(eval_bottom_keys):,}")
print(f"Existing in both: {len(existing_bottom):,}")
print(f"Eval cold-start series: {len(cold_start_bottom):,}")
print(f"Train-only series: {len(train_only_bottom):,}")

if cold_start_bottom:
    print(
        "Cold-start eval series are reported but excluded from this fixed-origin lag-based reconciliation experiment; "
        "inventing their missing demand history would be invalid."
    )

existing_index = pd.MultiIndex.from_tuples(sorted(existing_bottom), names=SERIES_KEYS)
train_existing_mask = pd.MultiIndex.from_frame(train_df[SERIES_KEYS]).isin(existing_index)
eval_existing_mask = pd.MultiIndex.from_frame(eval_df[SERIES_KEYS]).isin(existing_index)
train_h = train_df.loc[train_existing_mask].copy()
eval_h = eval_df.loc[eval_existing_mask].copy()

if train_h.empty or eval_h.empty:
    raise RuntimeError("No established bottom series remain for hierarchical evaluation.")

In [ ]:
LEVEL_SPECS: Mapping[str, List[str]] = {
    "Company": [],
    "City": ["city_id"],
    "Store": ["city_id", "store_id"],
    "Category": ["city_id", "store_id", "third_category_id"],
    "SKU": ["city_id", "store_id", "third_category_id", "product_id"],
}
LEVEL_ORDER = list(LEVEL_SPECS.keys())


def node_id_from_values(level: str, values: Sequence) -> str:
    if level == "Company":
        return "Company"
    return level + "|" + "|".join(str(v) for v in values)


def add_node_ids(mapping: pd.DataFrame) -> pd.DataFrame:
    out = mapping.copy()
    for level, cols in LEVEL_SPECS.items():
        if level == "Company":
            out[f"node_{level}"] = "Company"
        else:
            out[f"node_{level}"] = [
                node_id_from_values(level, vals)
                for vals in out[cols].itertuples(index=False, name=None)
            ]
    return out

bottom_mapping = add_node_ids(
    train_h[["city_id", "store_id", "third_category_id", "product_id"]].drop_duplicates()
)

node_frames = []
for level in LEVEL_ORDER:
    ids = bottom_mapping[[f"node_{level}"]].drop_duplicates().rename(columns={f"node_{level}": "node_id"})
    ids["level"] = level
    node_frames.append(ids)
node_metadata = pd.concat(node_frames, ignore_index=True)
node_metadata["level"] = pd.Categorical(node_metadata["level"], categories=LEVEL_ORDER, ordered=True)
node_metadata = node_metadata.sort_values(["level", "node_id"]).reset_index(drop=True)
node_metadata["row_idx"] = np.arange(len(node_metadata), dtype=np.int64)

node_counts = node_metadata.groupby("level", observed=False).size().rename("nodes").reset_index()
display(node_counts)

### Summing matrix `S`

Let `b` be the vector of bottom-level SKU forecasts. The hierarchy is coherent when the full forecast vector satisfies:

`y = S b`

Each bottom series contributes exactly once to its category, store, city, and company totals. The matrix is sparse: each bottom column has only five non-zero entries.

In [ ]:
def build_summing_matrix(bottom_map: pd.DataFrame, nodes: pd.DataFrame) -> Tuple[sparse.csr_matrix, List[str]]:
    row_lookup = dict(zip(nodes["node_id"], nodes["row_idx"]))
    sku_nodes = bottom_map["node_SKU"].tolist()
    col_lookup = {node: idx for idx, node in enumerate(sku_nodes)}

    rows: List[int] = []
    cols: List[int] = []
    data: List[float] = []
    for _, row in bottom_map.iterrows():
        j = col_lookup[row["node_SKU"]]
        for level in LEVEL_ORDER:
            rows.append(row_lookup[row[f"node_{level}"]])
            cols.append(j)
            data.append(1.0)

    S = sparse.coo_matrix((data, (rows, cols)), shape=(len(nodes), len(sku_nodes))).tocsr()
    return S, sku_nodes

S, bottom_node_order = build_summing_matrix(bottom_mapping, node_metadata)
node_order = node_metadata["node_id"].tolist()
node_index = {node: i for i, node in enumerate(node_order)}
bottom_index = {node: j for j, node in enumerate(bottom_node_order)}

assert S.shape == (len(node_order), len(bottom_node_order))
assert np.all(np.asarray(S.sum(axis=0)).ravel() == len(LEVEL_ORDER))
print(f"S shape={S.shape}; nnz={S.nnz:,}; density={S.nnz / (S.shape[0]*S.shape[1]):.6%}")

## 7. Hierarchical daily aggregation

Every hierarchy level receives its **own demand series** and will later receive an independently fitted forecasting model. This is what creates the possibility of incoherence: independently trained models have no reason to obey parent/child summation constraints.

In [ ]:
def aggregate_level(df: pd.DataFrame, level: str) -> pd.DataFrame:
    """Aggregate bottom demand into daily series for one hierarchy level."""
    keys = LEVEL_SPECS[level]
    group_cols = keys + ["dt"]
    agg = df.groupby(group_cols, observed=True, as_index=False)["sale_amount"].sum()
    if level == "Company":
        agg["series_code"] = "Company"
        agg["node_id"] = "Company"
    else:
        agg["series_code"] = [
            node_id_from_values(level, vals)
            for vals in agg[keys].itertuples(index=False, name=None)
        ]
        agg["node_id"] = agg["series_code"]
    agg["level"] = level
    return agg.sort_values(["series_code", "dt"]).reset_index(drop=True)

train_levels = {level: aggregate_level(train_h, level) for level in LEVEL_ORDER}
eval_levels = {level: aggregate_level(eval_h, level) for level in LEVEL_ORDER}

for level in LEVEL_ORDER:
    print(level, "train", train_levels[level].shape, "eval", eval_levels[level].shape)

### Daily-contiguity audit

Lag features represent calendar-day lags only when each hierarchy series is daily and contiguous. The notebook therefore checks observed date gaps instead of silently interpreting row offsets as day offsets.

In [ ]:
def gap_audit(level_df: pd.DataFrame) -> pd.DataFrame:
    ordered = level_df.sort_values(["series_code", "dt"]).copy()
    ordered["day_gap"] = ordered.groupby("series_code", observed=True)["dt"].diff().dt.days
    gaps = ordered.loc[ordered["day_gap"].notna() & ordered["day_gap"].ne(1), ["series_code", "dt", "day_gap"]]
    return gaps

bottom_gaps = gap_audit(train_levels["SKU"])
print(f"Non-daily SKU transitions in train: {len(bottom_gaps):,}")
if not bottom_gaps.empty:
    display(bottom_gaps.head(20))
    raise ValueError(
        "Bottom series are not daily-contiguous. Reconciliation forecasting is stopped because row-based lag features "
        "would no longer represent the stated day lags."
    )

## 8. Leakage-safe level forecasting features

Only calendar variables and lagged/rolling demand are used. No future target, same-day stock, realized weather, or same-day sales-derived variable enters the models.

All rolling statistics use `shift(1)`.

In [ ]:
CALENDAR_FEATURES = ["day_of_week", "day_of_month", "week_of_year", "month", "is_weekend"]
LAG_FEATURES = ["lag_1", "lag_7", "lag_14", "lag_28"]
ROLL_FEATURES = ["rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_std_28"]
MODEL_FEATURES = CALENDAR_FEATURES + LAG_FEATURES + ROLL_FEATURES + ["series_code"]
REQUIRED_HISTORY_FEATURES = LAG_FEATURES + ROLL_FEATURES


def create_level_features(level_df: pd.DataFrame) -> pd.DataFrame:
    out = level_df.sort_values(["series_code", "dt"]).copy()
    out["day_of_week"] = out["dt"].dt.dayofweek.astype("int8")
    out["day_of_month"] = out["dt"].dt.day.astype("int8")
    out["week_of_year"] = out["dt"].dt.isocalendar().week.astype("int16")
    out["month"] = out["dt"].dt.month.astype("int8")
    out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype("int8")

    g = out.groupby("series_code", sort=False, observed=True)["sale_amount"]
    for lag in (1, 7, 14, 28):
        out[f"lag_{lag}"] = g.shift(lag)
    for window in (7, 28):
        out[f"rolling_mean_{window}"] = g.transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=w).mean()
        )
        out[f"rolling_std_{window}"] = g.transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=w).std()
        )
    return out


def prepare_training_matrix(level_df: pd.DataFrame, cutoff: pd.Timestamp) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    feats = create_level_features(level_df.loc[level_df["dt"] <= cutoff])
    usable = feats.dropna(subset=REQUIRED_HISTORY_FEATURES).copy()
    if usable.empty:
        raise ValueError(f"No usable training rows through {cutoff} after 28-day history requirements.")
    categories = sorted(usable["series_code"].astype(str).unique().tolist())
    usable["series_code"] = pd.Categorical(usable["series_code"].astype(str), categories=categories)
    X = usable[MODEL_FEATURES]
    y = usable["sale_amount"].astype(float)
    assert "sale_amount" not in X.columns
    return X, y, categories

In [ ]:
LGBM_PARAMS = {
    "objective": "regression_l1",
    "learning_rate": 0.05,
    "n_estimators": 450,
    "num_leaves": 63,
    "min_child_samples": 40,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}


def train_level_model(level_df: pd.DataFrame, cutoff: pd.Timestamp) -> Tuple[lgb.LGBMRegressor, List[str]]:
    X, y, categories = prepare_training_matrix(level_df, cutoff)
    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    model.fit(X, y, categorical_feature=["series_code"])
    return model, categories


def _features_from_history(values: Sequence[float], target_date: pd.Timestamp) -> Dict[str, float]:
    arr = np.asarray(values, dtype=float)
    if len(arr) < 28:
        raise ValueError("At least 28 historical observations are required for recursive forecasting.")
    return {
        "day_of_week": int(target_date.dayofweek),
        "day_of_month": int(target_date.day),
        "week_of_year": int(target_date.isocalendar().week),
        "month": int(target_date.month),
        "is_weekend": int(target_date.dayofweek >= 5),
        "lag_1": float(arr[-1]),
        "lag_7": float(arr[-7]),
        "lag_14": float(arr[-14]),
        "lag_28": float(arr[-28]),
        "rolling_mean_7": float(arr[-7:].mean()),
        "rolling_mean_28": float(arr[-28:].mean()),
        "rolling_std_7": float(arr[-7:].std(ddof=1)),
        "rolling_std_28": float(arr[-28:].std(ddof=1)),
    }


def recursive_level_forecast(
    model: lgb.LGBMRegressor,
    categories: Sequence[str],
    history_df: pd.DataFrame,
    forecast_dates: Sequence[pd.Timestamp],
    level: str,
) -> pd.DataFrame:
    """Fixed-origin recursive 1..H forecast for every established node at one level."""
    cutoff = history_df["dt"].max()
    hist = (
        history_df.loc[history_df["dt"] <= cutoff]
        .sort_values(["series_code", "dt"])
        .groupby("series_code", observed=True)["sale_amount"]
        .apply(list)
        .to_dict()
    )
    node_ids = sorted(hist.keys())
    unknown = sorted(set(map(str, node_ids)) - set(map(str, categories)))
    if unknown:
        raise ValueError(f"{level}: prediction nodes unseen during model fitting: {unknown[:10]}")

    rows = []
    negative_raw = 0
    for h, date in enumerate(sorted(pd.to_datetime(forecast_dates)), start=1):
        records = []
        record_nodes = []
        for node in node_ids:
            feat = _features_from_history(hist[node], pd.Timestamp(date))
            feat["series_code"] = str(node)
            records.append(feat)
            record_nodes.append(node)
        X = pd.DataFrame(records)
        X["series_code"] = pd.Categorical(X["series_code"], categories=list(categories))
        raw_pred = model.predict(X[MODEL_FEATURES])
        negative_raw += int((raw_pred < 0).sum())
        pred = np.maximum(raw_pred, 0.0)
        for node, raw, yhat in zip(record_nodes, raw_pred, pred):
            hist[node].append(float(yhat))
            rows.append({
                "node_id": str(node),
                "level": level,
                "target_date": pd.Timestamp(date),
                "horizon": h,
                "base_forecast_raw": float(raw),
                "base_forecast": float(yhat),
            })
    print(f"{level}: raw negative forecasts before clipping = {negative_raw:,}")
    return pd.DataFrame(rows)

## 9. Chronological rolling-origin backtesting

The training split supplies three 7-day folds. For every fold and every hierarchy level:

1. features and models use dates only through `train_end`,
2. the model recursively forecasts the following seven dates,
3. actual validation data is used only after prediction,
4. residuals are retained for MinT-style variance estimation.

No random train/test split is used.

In [ ]:
def make_rolling_origin_folds(df: pd.DataFrame, horizon: int, n_folds: int) -> List[Dict[str, pd.Timestamp]]:
    dates = np.array(sorted(pd.to_datetime(df["dt"].unique())))
    needed = horizon * n_folds
    if len(dates) <= MIN_HISTORY_DAYS + needed:
        raise ValueError(
            f"Need more than {MIN_HISTORY_DAYS + needed} unique dates for lag history and {n_folds} folds; "
            f"found {len(dates)}."
        )
    first_val = len(dates) - needed
    folds = []
    for i in range(n_folds):
        val_start = first_val + i * horizon
        train_end = pd.Timestamp(dates[val_start - 1])
        val_dates = pd.to_datetime(dates[val_start: val_start + horizon])
        folds.append({
            "fold": i + 1,
            "train_start": pd.Timestamp(dates[0]),
            "train_end": train_end,
            "validation_start": pd.Timestamp(val_dates[0]),
            "validation_end": pd.Timestamp(val_dates[-1]),
        })
    return folds

folds = make_rolling_origin_folds(train_h, FORECAST_HORIZON, N_BACKTEST_FOLDS)
fold_table = pd.DataFrame(folds)
assert (fold_table["train_end"] < fold_table["validation_start"]).all()
display(fold_table)

In [ ]:
def actual_level_rows(level_df: pd.DataFrame, dates: Sequence[pd.Timestamp], level: str) -> pd.DataFrame:
    actual = level_df.loc[level_df["dt"].isin(pd.to_datetime(dates)), ["series_code", "dt", "sale_amount"]].copy()
    actual = actual.rename(columns={"series_code": "node_id", "dt": "target_date", "sale_amount": "actual"})
    actual["node_id"] = actual["node_id"].astype(str)
    actual["level"] = level
    origin = pd.Timestamp(min(pd.to_datetime(dates))) - pd.Timedelta(days=1)
    actual["horizon"] = (actual["target_date"] - origin).dt.days.astype(int)
    return actual

backtest_rows = []
folds_to_run = folds if RUN_FULL_BACKTEST else folds[-1:]
for fold in folds_to_run:
    val_dates = pd.date_range(fold["validation_start"], fold["validation_end"], freq="D")
    for level in LEVEL_ORDER:
        level_train = train_levels[level]
        model, categories = train_level_model(level_train, fold["train_end"])
        pred = recursive_level_forecast(
            model,
            categories,
            level_train.loc[level_train["dt"] <= fold["train_end"]],
            val_dates,
            level,
        )
        actual = actual_level_rows(level_train, val_dates, level)
        merged = pred.merge(actual, on=["node_id", "level", "target_date", "horizon"], how="inner", validate="one_to_one")
        merged["residual"] = merged["actual"] - merged["base_forecast"]
        merged["fold"] = fold["fold"]
        merged["training_cutoff"] = fold["train_end"]
        assert (merged["training_cutoff"] < merged["target_date"]).all()
        backtest_rows.append(merged)
        del model, pred, actual, merged
        gc.collect()

backtest_predictions = pd.concat(backtest_rows, ignore_index=True)
print("Backtest prediction rows:", len(backtest_predictions))
display(backtest_predictions.head())

## 10. Forecast metrics

Metrics match the conventions established earlier in DemandPulse:

- **MAE** — mean absolute error
- **RMSE** — root mean squared error
- **WAPE** — `sum(|y - ŷ|) / sum(|y|)`
- **Bias** — `sum(ŷ - y) / sum(|y|)`; positive means net over-forecasting

Every denominator is protected against zero.

In [ ]:
def calculate_metrics(y_true: Sequence[float], y_pred: Sequence[float]) -> Dict[str, float]:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p)
    y, p = y[mask], p[mask]
    if len(y) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "WAPE": np.nan, "Bias": np.nan, "observations": 0}
    denom = np.abs(y).sum()
    return {
        "MAE": float(mean_absolute_error(y, p)),
        "RMSE": float(np.sqrt(mean_squared_error(y, p))),
        "WAPE": float(np.abs(y - p).sum() / denom) if denom > 0 else np.nan,
        "Bias": float((p - y).sum() / denom) if denom > 0 else np.nan,
        "observations": int(len(y)),
    }

base_bt_metrics = []
for (fold, level), grp in backtest_predictions.groupby(["fold", "level"], observed=True):
    base_bt_metrics.append({"fold": fold, "level": level, **calculate_metrics(grp["actual"], grp["base_forecast"])})
base_bt_metrics = pd.DataFrame(base_bt_metrics)
display(base_bt_metrics)

## 11. Reconciliation ingredients estimated from training only

### Historical proportions

Top-down and middle-out require allocation proportions. They are estimated from the trailing `PROPORTION_WINDOW_DAYS` of training data before the forecast origin.

### MinT-style scale matrix

Full MinT requires a potentially enormous cross-series residual covariance matrix. With thousands of bottom series, dense covariance estimation/inversion is neither statistically stable nor memory-conscious.

This notebook therefore implements a **MinT-style diagonal WLS approximation**:

- backtest residual mean-squared error estimates one variance scale per node,
- missing/unstable scales shrink to a level median and then a global median,
- a sparse weighted projection finds the coherent forecast closest to all independent forecasts under inverse-variance weighting.

The approximation is explicit and is not misrepresented as full-covariance MinT.

In [ ]:
def trailing_bottom_shares(
    train_bottom_df: pd.DataFrame,
    cutoff: pd.Timestamp,
    bottom_map: pd.DataFrame,
    window_days: int,
) -> pd.DataFrame:
    start = cutoff - pd.Timedelta(days=window_days - 1)
    recent = train_bottom_df.loc[train_bottom_df["dt"].between(start, cutoff)].copy()
    totals = recent.groupby(SERIES_KEYS, observed=True)["sale_amount"].sum().rename("recent_demand").reset_index()
    shares = bottom_map.merge(totals, on=SERIES_KEYS, how="left")
    shares["recent_demand"] = shares["recent_demand"].fillna(0.0)

    total = shares["recent_demand"].sum()
    if total > 0:
        shares["company_share"] = shares["recent_demand"] / total
    else:
        shares["company_share"] = 1.0 / len(shares)

    store_total = shares.groupby("node_Store", observed=True)["recent_demand"].transform("sum")
    store_n = shares.groupby("node_Store", observed=True)["recent_demand"].transform("size")
    shares["store_share"] = np.where(store_total > 0, shares["recent_demand"] / store_total, 1.0 / store_n)

    assert np.isclose(shares["company_share"].sum(), 1.0)
    check = shares.groupby("node_Store", observed=True)["store_share"].sum()
    assert np.allclose(check.to_numpy(), 1.0)
    return shares


def estimate_node_variances(backtest: pd.DataFrame, nodes: pd.DataFrame) -> np.ndarray:
    stats = backtest.groupby(["node_id", "level"], observed=True)["residual"].agg(
        mse=lambda s: float(np.mean(np.square(s))), n="size"
    ).reset_index()
    level_medians = stats.loc[stats["mse"] > 0].groupby("level", observed=True)["mse"].median().to_dict()
    positive = stats.loc[np.isfinite(stats["mse"]) & (stats["mse"] > 0), "mse"]
    global_median = float(positive.median()) if len(positive) else 1.0
    stat_map = dict(zip(stats["node_id"], stats["mse"]))

    variances = []
    for row in nodes.itertuples(index=False):
        value = stat_map.get(row.node_id, np.nan)
        if not np.isfinite(value) or value <= 0:
            value = level_medians.get(str(row.level), global_median)
        if not np.isfinite(value) or value <= 0:
            value = global_median
        variances.append(max(float(value), 1e-8))
    return np.asarray(variances, dtype=float)

node_variances = estimate_node_variances(backtest_predictions, node_metadata)
print(pd.Series(node_variances).describe())

## 12. Reconciliation methods

For each forecast horizon, all methods start from the same independent base forecast vector.

- **Bottom-up:** trust SKU forecasts; aggregate upward with `S`.
- **Top-down:** trust the independent company forecast; allocate it to SKUs using historical company shares; aggregate upward.
- **Middle-out:** trust independent store forecasts; allocate each store to its SKUs using within-store historical shares; aggregate upward.
- **MinT-style:** use all independent levels simultaneously in a residual-variance-weighted coherent projection.

In [ ]:
def vector_from_frame(frame: pd.DataFrame, value_col: str) -> np.ndarray:
    lookup = frame.set_index("node_id")[value_col]
    missing = [n for n in node_order if n not in lookup.index]
    if missing:
        raise ValueError(f"Forecast vector missing {len(missing)} hierarchy nodes; examples: {missing[:10]}")
    return lookup.reindex(node_order).to_numpy(dtype=float)


def bottom_up(base_vector: np.ndarray) -> np.ndarray:
    bottom_rows = node_metadata.loc[node_metadata["level"].eq("SKU"), "row_idx"].to_numpy()
    bottom = base_vector[bottom_rows]
    return np.asarray(S @ bottom).ravel()


def top_down(base_vector: np.ndarray, shares: pd.DataFrame) -> np.ndarray:
    company_forecast = float(base_vector[node_index["Company"]])
    bottom = np.zeros(len(bottom_node_order), dtype=float)
    share_map = dict(zip(shares["node_SKU"], shares["company_share"]))
    for node, j in bottom_index.items():
        bottom[j] = company_forecast * float(share_map[node])
    return np.asarray(S @ bottom).ravel()


def middle_out_store(base_vector: np.ndarray, shares: pd.DataFrame) -> np.ndarray:
    bottom = np.zeros(len(bottom_node_order), dtype=float)
    share_map = shares.set_index("node_SKU")[["node_Store", "store_share"]]
    for sku_node, j in bottom_index.items():
        store_node = share_map.loc[sku_node, "node_Store"]
        store_share = float(share_map.loc[sku_node, "store_share"])
        bottom[j] = float(base_vector[node_index[store_node]]) * store_share
    return np.asarray(S @ bottom).ravel()


def mint_style_diagonal(
    base_vector: np.ndarray,
    variances: np.ndarray,
    ridge: float = MINT_RIDGE,
    enforce_nonnegative: bool = True,
) -> Tuple[np.ndarray, Dict[str, float]]:
    """Sparse diagonal-W MinT-style reconciliation solved in bottom space with conjugate gradient."""
    if len(base_vector) != S.shape[0] or len(variances) != S.shape[0]:
        raise ValueError("Base vector/variance vector does not match summing matrix rows.")
    w_inv = 1.0 / np.maximum(variances, 1e-8)

    def matvec(v):
        return np.asarray(S.T @ (w_inv * np.asarray(S @ v).ravel())).ravel() + ridge * v

    A = LinearOperator((S.shape[1], S.shape[1]), matvec=matvec, dtype=float)
    b = np.asarray(S.T @ (w_inv * base_vector)).ravel()
    x, info = cg(A, b, atol=0.0, rtol=1e-6, maxiter=2_000)
    if info != 0:
        raise RuntimeError(f"MinT-style conjugate-gradient solve did not converge; info={info}")

    negative_bottom = int((x < 0).sum())
    raw_min = float(x.min()) if len(x) else np.nan
    operational_bottom = np.maximum(x, 0.0) if enforce_nonnegative else x
    coherent = np.asarray(S @ operational_bottom).ravel()
    diagnostics = {"negative_bottom_before_clip": negative_bottom, "minimum_bottom_before_clip": raw_min}
    return coherent, diagnostics

## 13. Final independent models and official fixed-origin forecasts

Model/reconciliation choices are now frozen. The official eval split is used only for final predictions and scoring.

The fixed forecast origin is `max(train date)`. If eval contains more than seven days, those dates are explicitly reported; only horizons 1–7 belong to this experiment.

In [ ]:
origin_date = train_h["dt"].max()
eval_h = eval_h.copy()
eval_h["horizon"] = (eval_h["dt"] - origin_date).dt.days
all_eval_horizons = sorted(eval_h["horizon"].dropna().unique().tolist())
print("Official eval horizons relative to train origin:", all_eval_horizons)

outside = eval_h.loc[~eval_h["horizon"].between(1, FORECAST_HORIZON)]
print(f"Eval rows outside fixed {FORECAST_HORIZON}-day experiment: {len(outside):,}")

valid_eval = eval_h.loc[eval_h["horizon"].between(1, FORECAST_HORIZON)].copy()
forecast_dates = pd.date_range(origin_date + pd.Timedelta(days=1), periods=FORECAST_HORIZON, freq="D")
missing_dates = sorted(set(forecast_dates) - set(valid_eval["dt"].unique()))
if missing_dates:
    raise ValueError(f"Official eval does not contain all required forecast dates: {missing_dates}")

In [ ]:
final_base_rows = []
final_models: Dict[str, lgb.LGBMRegressor] = {}
final_categories: Dict[str, List[str]] = {}

for level in LEVEL_ORDER:
    model, categories = train_level_model(train_levels[level], origin_date)
    final_models[level] = model
    final_categories[level] = categories
    preds = recursive_level_forecast(model, categories, train_levels[level], forecast_dates, level)
    final_base_rows.append(preds)
    model.booster_.save_model(str(HIER_MODELS_DIR / f"base_{level.lower()}.txt"))

base_forecasts = pd.concat(final_base_rows, ignore_index=True)
print("Independent base forecast rows:", len(base_forecasts))
display(base_forecasts.head())

### Why independent forecasts are incoherent

A store model and the models for its SKUs were fitted independently. Therefore:

`independent store forecast ≠ sum(independent SKU forecasts)`

The next cell finds a store with a large discrepancy and displays it directly.

In [ ]:
def find_store_incoherence_example(base: pd.DataFrame, bottom_map: pd.DataFrame) -> pd.DataFrame:
    sku_to_store = bottom_map[["node_SKU", "node_Store"]].drop_duplicates()
    sku = base.loc[base["level"].eq("SKU")].merge(
        sku_to_store, left_on="node_id", right_on="node_SKU", how="left", validate="many_to_one"
    )
    sku_sum = sku.groupby(["node_Store", "target_date", "horizon"], observed=True)["base_forecast"].sum().rename("sum_sku_forecast").reset_index()
    store = base.loc[base["level"].eq("Store"), ["node_id", "target_date", "horizon", "base_forecast"]].rename(
        columns={"node_id": "node_Store", "base_forecast": "independent_store_forecast"}
    )
    comp = store.merge(sku_sum, on=["node_Store", "target_date", "horizon"], how="inner", validate="one_to_one")
    comp["absolute_incoherence"] = (comp["independent_store_forecast"] - comp["sum_sku_forecast"]).abs()
    example_store = comp.groupby("node_Store", observed=True)["absolute_incoherence"].mean().idxmax()
    return comp.loc[comp["node_Store"].eq(example_store)].sort_values("target_date")

store_example = find_store_incoherence_example(base_forecasts, bottom_mapping)
display(store_example)

## 14. Apply reconciliation to the official eval horizon

Historical proportions are estimated **once from training through the forecast origin**. MinT-style variance scales come only from training backtest residuals.

In [ ]:
final_shares = trailing_bottom_shares(train_h, origin_date, bottom_mapping, PROPORTION_WINDOW_DAYS)

reconciled_rows = []
mint_diagnostics = []
for h, target_date in enumerate(forecast_dates, start=1):
    day_base = base_forecasts.loc[base_forecasts["target_date"].eq(target_date)]
    base_vec = vector_from_frame(day_base, "base_forecast")

    method_vectors = {
        "Unreconciled": base_vec,
        "Bottom-Up": bottom_up(base_vec),
        "Top-Down": top_down(base_vec, final_shares),
        "Middle-Out": middle_out_store(base_vec, final_shares),
    }
    mint_vec, mint_diag = mint_style_diagonal(base_vec, node_variances)
    method_vectors["MinT-Style"] = mint_vec
    mint_diagnostics.append({"horizon": h, "target_date": target_date, **mint_diag})

    for method, vec in method_vectors.items():
        tmp = node_metadata[["node_id", "level", "row_idx"]].copy()
        tmp["forecast"] = vec[tmp["row_idx"].to_numpy()]
        tmp["method"] = method
        tmp["target_date"] = target_date
        tmp["horizon"] = h
        reconciled_rows.append(tmp.drop(columns="row_idx"))

all_method_forecasts = pd.concat(reconciled_rows, ignore_index=True)
mint_diagnostics = pd.DataFrame(mint_diagnostics)
display(mint_diagnostics)

## 15. Official actual hierarchy and common evaluation rows

Actual eval demand is aggregated through the **same supported bottom-series hierarchy**. This ensures every method is scored against the same nodes and dates.

In [ ]:
valid_eval_levels = {level: aggregate_level(valid_eval, level) for level in LEVEL_ORDER}
actual_rows = []
for level in LEVEL_ORDER:
    frame = valid_eval_levels[level][["node_id", "level", "dt", "sale_amount"]].copy()
    frame = frame.rename(columns={"dt": "target_date", "sale_amount": "actual"})
    frame["horizon"] = (frame["target_date"] - origin_date).dt.days.astype(int)
    actual_rows.append(frame)
actual_hierarchy = pd.concat(actual_rows, ignore_index=True)

hierarchical_forecasts = all_method_forecasts.merge(
    actual_hierarchy,
    on=["node_id", "level", "target_date", "horizon"],
    how="inner",
    validate="many_to_one",
)

expected_rows = len(node_metadata) * FORECAST_HORIZON * all_method_forecasts["method"].nunique()
print(f"Expected common forecast rows: {expected_rows:,}")
print(f"Rows with valid actuals and forecasts: {len(hierarchical_forecasts):,}")
if len(hierarchical_forecasts) != expected_rows:
    raise ValueError(
        "Not every method has a complete common evaluation grid. Refusing to compare methods on different observations."
    )

## 16. Coherence diagnostics

For any forecast vector `y`, its bottom-implied coherent counterpart is `S × y_bottom`.

The notebook reports:

- mean absolute coherence discrepancy,
- maximum discrepancy,
- normalized coherence error,
- number of aggregate nodes violating the tolerance.

Bottom-level nodes have zero self-discrepancy by construction; coherence is assessed on aggregate nodes.

In [ ]:
def coherence_diagnostics(forecast_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    aggregate_mask = ~node_metadata["level"].eq("SKU").to_numpy()
    bottom_rows = node_metadata.loc[node_metadata["level"].eq("SKU"), "row_idx"].to_numpy()

    for (method, h, date), grp in forecast_frame.groupby(["method", "horizon", "target_date"], observed=True):
        vec = vector_from_frame(grp, "forecast")
        bottom = vec[bottom_rows]
        implied = np.asarray(S @ bottom).ravel()
        diff = vec - implied
        agg_diff = diff[aggregate_mask]
        agg_implied = implied[aggregate_mask]
        denom = np.abs(agg_implied).sum()
        rows.append({
            "method": method,
            "horizon": int(h),
            "target_date": date,
            "mean_abs_coherence_error": float(np.abs(agg_diff).mean()),
            "max_abs_coherence_error": float(np.abs(agg_diff).max()),
            "normalized_coherence_error": float(np.abs(agg_diff).sum() / denom) if denom > 0 else np.nan,
            "violated_aggregate_nodes": int((np.abs(agg_diff) > COHERENCE_TOLERANCE).sum()),
            "aggregate_nodes": int(aggregate_mask.sum()),
        })
    return pd.DataFrame(rows)

coherence_analysis = coherence_diagnostics(all_method_forecasts)
display(coherence_analysis.groupby("method", observed=True)[[
    "mean_abs_coherence_error", "normalized_coherence_error", "violated_aggregate_nodes"
]].mean())

## 17. Accuracy by hierarchy level and method

Reconciliation can improve some levels while worsening others. That trade-off is the core scientific question; zero coherence error alone is not evidence of a better forecast.

In [ ]:
level_metric_rows = []
for (method, level), grp in hierarchical_forecasts.groupby(["method", "level"], observed=True):
    metrics = calculate_metrics(grp["actual"], grp["forecast"])

    coherence_sub = []
    for (h, date), day_grp in grp.groupby(["horizon", "target_date"], observed=True):
        all_day = all_method_forecasts.loc[
            all_method_forecasts["method"].eq(method)
            & all_method_forecasts["horizon"].eq(h)
            & all_method_forecasts["target_date"].eq(date)
        ]
        vec = vector_from_frame(all_day, "forecast")
        bottom_rows = node_metadata.loc[node_metadata["level"].eq("SKU"), "row_idx"].to_numpy()
        implied = np.asarray(S @ vec[bottom_rows]).ravel()
        level_rows = node_metadata.loc[node_metadata["level"].eq(level), "row_idx"].to_numpy()
        coherence_sub.extend(np.abs(vec[level_rows] - implied[level_rows]).tolist())

    level_metric_rows.append({
        "method": method,
        "level": level,
        **metrics,
        "coherence_error": float(np.mean(coherence_sub)) if coherence_sub else np.nan,
    })

hierarchy_level_metrics = pd.DataFrame(level_metric_rows)
hierarchy_level_metrics["level"] = pd.Categorical(hierarchy_level_metrics["level"], categories=LEVEL_ORDER, ordered=True)
hierarchy_level_metrics = hierarchy_level_metrics.sort_values(["level", "WAPE"]).reset_index(drop=True)
display(hierarchy_level_metrics)

In [ ]:
reconciliation_rows = []
for method, grp in hierarchical_forecasts.groupby("method", observed=True):
    metrics = calculate_metrics(grp["actual"], grp["forecast"])
    coh = coherence_analysis.loc[coherence_analysis["method"].eq(method)]
    reconciliation_rows.append({
        "method": method,
        **metrics,
        "mean_abs_coherence_error": float(coh["mean_abs_coherence_error"].mean()),
        "normalized_coherence_error": float(coh["normalized_coherence_error"].mean()),
        "violated_aggregate_nodes_mean": float(coh["violated_aggregate_nodes"].mean()),
    })

reconciliation_results = pd.DataFrame(reconciliation_rows).sort_values("WAPE").reset_index(drop=True)
display(reconciliation_results)

## 18. Horizon-specific accuracy/coherence trade-off

A reconciliation method may behave differently at horizon 1 than at horizon 7. The next table retains the chronological horizon dimension rather than averaging it away.

In [ ]:
horizon_rows = []
for (method, h), grp in hierarchical_forecasts.groupby(["method", "horizon"], observed=True):
    metrics = calculate_metrics(grp["actual"], grp["forecast"])
    coh = coherence_analysis.loc[
        coherence_analysis["method"].eq(method) & coherence_analysis["horizon"].eq(h)
    ]
    horizon_rows.append({
        "method": method,
        "horizon": int(h),
        **metrics,
        "mean_abs_coherence_error": float(coh["mean_abs_coherence_error"].mean()),
        "normalized_coherence_error": float(coh["normalized_coherence_error"].mean()),
    })
horizon_metrics = pd.DataFrame(horizon_rows)
display(horizon_metrics.sort_values(["horizon", "WAPE"]))

## 19. Useful hierarchy visualizations

The plots focus on structure, incoherence, accuracy, and reconciliation adjustments rather than decorative charting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(node_counts["level"].astype(str), node_counts["nodes"])
ax.set_title("Hierarchy size by level")
ax.set_xlabel("Hierarchy level")
ax.set_ylabel("Number of nodes")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hierarchy_node_counts.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(store_example["target_date"], store_example["independent_store_forecast"], marker="o", label="Independent Store")
ax.plot(store_example["target_date"], store_example["sum_sku_forecast"], marker="o", label="Sum of independent SKUs")
ax.set_title(f"Example incoherence: {store_example['node_Store'].iloc[0]}")
ax.set_xlabel("Forecast date")
ax.set_ylabel("Forecast demand")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "store_incoherence_example.png", dpi=150)
plt.show()

In [ ]:
pivot_wape = hierarchy_level_metrics.pivot(index="level", columns="method", values="WAPE").reindex(LEVEL_ORDER)
fig, ax = plt.subplots(figsize=(10, 5))
for method in pivot_wape.columns:
    ax.plot(pivot_wape.index.astype(str), pivot_wape[method], marker="o", label=method)
ax.set_title("WAPE by hierarchy level and reconciliation method")
ax.set_xlabel("Hierarchy level")
ax.set_ylabel("WAPE")
ax.legend(ncol=2)
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hierarchy_wape_by_method.png", dpi=150)
plt.show()

In [ ]:
coh_summary = coherence_analysis.groupby("method", observed=True)["normalized_coherence_error"].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(coh_summary.index, coh_summary.values)
ax.set_title("Normalized coherence error by method")
ax.set_xlabel("Method")
ax.set_ylabel("Normalized coherence error")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "coherence_error_by_method.png", dpi=150)
plt.show()

In [ ]:
base_only = all_method_forecasts.loc[all_method_forecasts["method"].eq("Unreconciled"), ["node_id", "target_date", "forecast"]].rename(columns={"forecast": "base"})
adjustments = all_method_forecasts.loc[~all_method_forecasts["method"].eq("Unreconciled")].merge(
    base_only, on=["node_id", "target_date"], how="left", validate="many_to_one"
)
adjustments = adjustments.merge(node_metadata[["node_id", "level"]], on="node_id", how="left", suffixes=("", "_meta"))
adjustments["adjustment"] = adjustments["forecast"] - adjustments["base"]

plot_data = [
    adjustments.loc[adjustments["level"].eq(level), "adjustment"].dropna().to_numpy()
    for level in LEVEL_ORDER
]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(plot_data, tick_labels=LEVEL_ORDER, showfliers=False)
ax.set_title("Distribution of reconciliation adjustments by hierarchy level")
ax.set_xlabel("Hierarchy level")
ax.set_ylabel("Reconciled forecast − independent forecast")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "reconciliation_adjustments_by_level.png", dpi=150)
plt.show()

## 20. Which levels improve and which become worse?

The following table expresses each reconciled method's WAPE change relative to the unreconciled forecast at the **same level and same observations**.

Negative `WAPE_change` means improved WAPE; positive means worse.

In [ ]:
base_level = hierarchy_level_metrics.loc[hierarchy_level_metrics["method"].eq("Unreconciled"), ["level", "WAPE", "MAE", "RMSE", "Bias"]].rename(
    columns={"WAPE": "base_WAPE", "MAE": "base_MAE", "RMSE": "base_RMSE", "Bias": "base_Bias"}
)
tradeoff = hierarchy_level_metrics.merge(base_level, on="level", how="left", validate="many_to_one")
tradeoff["WAPE_change"] = tradeoff["WAPE"] - tradeoff["base_WAPE"]
tradeoff["MAE_change"] = tradeoff["MAE"] - tradeoff["base_MAE"]
tradeoff["improved_WAPE"] = tradeoff["WAPE_change"] < 0
tradeoff = tradeoff.loc[~tradeoff["method"].eq("Unreconciled")].sort_values(["method", "level"])
display(tradeoff[["method", "level", "WAPE", "base_WAPE", "WAPE_change", "MAE_change", "coherence_error", "improved_WAPE"]])

## 21. Save required artifacts

`hierarchical_forecasts.csv` stores the common official-eval prediction grid across methods. The remaining outputs are compact comparison tables.

In [ ]:
HIERARCHICAL_FORECASTS_PATH = TABLES_DIR / "hierarchical_forecasts.csv"
RECONCILIATION_RESULTS_PATH = TABLES_DIR / "reconciliation_results.csv"
LEVEL_METRICS_PATH = TABLES_DIR / "hierarchy_level_metrics.csv"
COHERENCE_ANALYSIS_PATH = TABLES_DIR / "coherence_analysis.csv"

hierarchical_forecasts.to_csv(HIERARCHICAL_FORECASTS_PATH, index=False)
reconciliation_results.to_csv(RECONCILIATION_RESULTS_PATH, index=False)
hierarchy_level_metrics.to_csv(LEVEL_METRICS_PATH, index=False)
coherence_analysis.to_csv(COHERENCE_ANALYSIS_PATH, index=False)

print("Saved:")
for p in [HIERARCHICAL_FORECASTS_PATH, RECONCILIATION_RESULTS_PATH, LEVEL_METRICS_PATH, COHERENCE_ANALYSIS_PATH]:
    print(" -", p)

## 22. Scientific findings

This section is generated from executed metrics. It does not assume that reconciliation must improve accuracy.

In [ ]:
base_overall = reconciliation_results.loc[reconciliation_results["method"].eq("Unreconciled")].iloc[0]
best_reconciled = reconciliation_results.loc[~reconciliation_results["method"].eq("Unreconciled")].sort_values("WAPE").iloc[0]

coh_base = float(base_overall["normalized_coherence_error"])
coh_best = float(best_reconciled["normalized_coherence_error"])
coh_reduction = 100 * (coh_base - coh_best) / coh_base if np.isfinite(coh_base) and coh_base > 0 else np.nan

level_changes = tradeoff.loc[tradeoff["method"].eq(best_reconciled["method"])].copy()
improved_levels = level_changes.loc[level_changes["WAPE_change"] < 0, "level"].astype(str).tolist()
worse_levels = level_changes.loc[level_changes["WAPE_change"] > 0, "level"].astype(str).tolist()

best_wape_change = float(best_reconciled["WAPE"] - base_overall["WAPE"])
accuracy_direction = "improved" if best_wape_change < 0 else "worsened" if best_wape_change > 0 else "was unchanged"

previous_context = ""
if "Notebook 01 final comparison" in previous_results:
    prev = previous_results["Notebook 01 final comparison"]
    if {"model", "WAPE"}.issubset(prev.columns) and prev["model"].eq("LightGBM").any():
        previous_wape = float(prev.loc[prev["model"].eq("LightGBM"), "WAPE"].iloc[0])
        previous_context = (
            f" Notebook 01's saved LightGBM WAPE was {previous_wape:.4f}; this is shown only as project context, "
            "not as an apples-to-apples hierarchical comparison unless its evaluation rows exactly match this notebook."
        )

findings = f"""
1. **Independent forecasts were incoherent:** the unreconciled normalized coherence error was {coh_base:.6f}. The store example above demonstrates directly that independently forecast store totals need not equal the sum of independently forecast SKUs.
2. **Best reconciled method by overall WAPE:** `{best_reconciled['method']}` achieved WAPE {best_reconciled['WAPE']:.4f} versus {base_overall['WAPE']:.4f} unreconciled, so overall WAPE {accuracy_direction} by {abs(best_wape_change):.4f}.
3. **Consistency:** `{best_reconciled['method']}` had normalized coherence error {coh_best:.6e}. The relative coherence-error reduction versus the base forecast was {coh_reduction:.2f}% when that ratio was defined.
4. **Levels improved under the best-WAPE reconciler:** {', '.join(improved_levels) if improved_levels else 'none'}.
5. **Levels that became worse under the best-WAPE reconciler:** {', '.join(worse_levels) if worse_levels else 'none'}.
6. **Accuracy/coherence trade-off:** reconciliation enforces or nearly enforces accounting consistency, but the level tables show that this can redistribute forecast error rather than uniformly reduce it. A method should therefore be selected using both accuracy and coherence, not coherence alone.
7. **MinT interpretation:** `MinT-Style` is a diagonal residual-variance WLS approximation estimated from training backtests. It uses information from every independently forecast level but does not estimate the full residual covariance matrix.
8. **Sampling interpretation:** with `SERIES_SAMPLE_PERCENT={SERIES_SAMPLE_PERCENT}`, the Company node is the total of the deterministic sampled hierarchy, not necessarily the retailer's full-company sales total.{previous_context}
"""
display(Markdown(findings))

## 23. Limitations

- `sale_amount` is a normalized observed-sales target, not an absolute unit-demand measure.
- Stockouts can censor observed sales, so aggregation does not recover latent unmet demand.
- With `SERIES_SAMPLE_PERCENT < 100`, the hierarchy represents the deterministic sampled portfolio; its Company/City/Store totals are not full-retailer totals where excluded series exist.
- The strict hierarchy uses **store-local `third_category_id`** nodes. A cross-store corporate category hierarchy would require a different multi-dimensional/grouped hierarchy design.
- Cold-start eval series are reported and excluded from the lag-based fixed-origin experiment rather than assigned invented history.
- Independent level forecasts use the same LightGBM family and feature policy but are still separate statistical models; this is sufficient to study incoherence but is not an exhaustive base-model search.
- Top-down and middle-out historical shares assume recent allocation patterns are informative for the forecast horizon.
- The MinT-style method uses a **diagonal residual-variance approximation**, not a full covariance MinT estimator. Full covariance is expensive and unstable with thousands of series and only a small number of backtest origins.
- Reconciliation is applied to point forecasts only. Probabilistic hierarchical reconciliation would require extending Notebook 02's uncertainty framework.
- Recursive 7-day forecasting does not propagate a full predictive distribution.
- Reconciliation can improve coherence while worsening accuracy at specific levels; no method is guaranteed to dominate.
- Results are retailer- and hierarchy-specific and may not generalize to unrelated retail systems.
- This notebook does not optimize inventory or service levels; that remains a separate decision problem.

## 24. Final artifact and consistency audit

In [ ]:
mandatory_artifacts = [
    HIERARCHICAL_FORECASTS_PATH,
    RECONCILIATION_RESULTS_PATH,
    LEVEL_METRICS_PATH,
    COHERENCE_ANALYSIS_PATH,
]
artifact_audit = pd.DataFrame({
    "artifact": [str(p) for p in mandatory_artifacts],
    "exists": [p.exists() for p in mandatory_artifacts],
    "size_bytes": [p.stat().st_size if p.exists() else 0 for p in mandatory_artifacts],
})
display(artifact_audit)
assert artifact_audit["exists"].all(), "One or more mandatory hierarchical artifacts were not written."

# Logical quality checks.
assert STRICT_TEMPORAL_HOLDOUT
assert "sale_amount" not in MODEL_FEATURES
assert all(col not in MODEL_FEATURES for col in ["stock_hour6_22_cnt", "discount", "holiday_flag", "activity_flag"])
assert (fold_table["train_end"] < fold_table["validation_start"]).all()
assert len(S.shape) == 2 and S.shape[1] == len(bottom_node_order)
assert set(all_method_forecasts["method"].unique()) == {"Unreconciled", "Bottom-Up", "Top-Down", "Middle-Out", "MinT-Style"}
assert hierarchical_forecasts[["actual", "forecast"]].notna().all().all()

reconciled_coherence = coherence_analysis.loc[~coherence_analysis["method"].eq("Unreconciled"), "max_abs_coherence_error"]
if not (reconciled_coherence <= 1e-6).all():
    raise AssertionError("A reconciled method is not numerically coherent within tolerance.")

print("Final quality audit passed.")
print("Notebook 04 complete.")
print("Completed:")
print("✓ Multi-level hierarchy")
print("✓ Independent forecasts")
print("✓ Bottom-up reconciliation")
print("✓ Top-down reconciliation")
print("✓ Middle-out reconciliation")
print("✓ MinT-style reconciliation")
print("✓ Accuracy and coherence evaluation")
print("✓ Scientific findings and limitations")
print("\nNext:\nNotebook 05 — Inventory Decision Optimization")